# Stateful Chatbot App


[Step 11 - Stateful chatbot app]

> **MLCourse - Agentic AI - Memory and State**

> Stage in the capstone: the capstone chatbot remembers prior turns PER THREAD thanks to this.

Time to assemble the parts into an APPLICATION: a persona prompt, a guarded
model, a LangGraph app with a checkpointer, and an interactive loop - the exact
shape the Step 12 capstone reuses with RAG bolted on.

> **Note on the API.** The old `RunnableWithMessageHistory` wrapper is
> deprecated; LangGraph persistence replaces it. Practically that means a
> `thread_id` instead of a `session_id`, and a *checkpointer* instead of a
> history-object factory. Everything else about the app is unchanged.

# What you will learn

1. Compose the chatbot: system persona + MessagesPlaceholder("history") + human input.
2. Wrap it in a LangGraph app whose `InMemorySaver` keeps one transcript per thread.
3. Run an input() loop that is QA-SAFE: MAX_TURNS caps it, 'quit' exits, and a
   closed stdin (EOFError) ends it cleanly instead of hanging automation.
4. Prove per-thread separation with two scripted conversations.
5. Export every transcript to DATA / "transcripts.json" for inspection.

### Sections

1. Setup
2. The librarian chain + LangGraph memory
3. The capped interactive loop
4. Two scripted threads - separation proof
5. Transcript export
6. Summary

### Section 1: setup


In [ ]:
from pathlib import Path          # cross-platform paths
import os                         # environment access
import json                       # transcripts export at the end
import re                         # offline stub helpers

def _find_track(start_dir):
    """Climb parent folders until we find (or reach) the dir named 03_agentic_ai."""
    here = Path(start_dir).resolve()
    for candidate in (here, *here.parents):
        if candidate.name == "03_agentic_ai":
            return candidate
        if (candidate / "03_agentic_ai").is_dir():
            return candidate / "03_agentic_ai"
    raise FileNotFoundError("Could not locate the 03_agentic_ai track near %s" % here)

TRACK = _find_track(Path.cwd())
DATA = TRACK / "data"
DATA.mkdir(parents=True, exist_ok=True)

from dotenv import load_dotenv    # keys are optional here; guards handle absence
load_dotenv(TRACK / ".env", override=False)
load_dotenv(override=False)

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("track:", TRACK)
print("data :", DATA)


### Section 2: the librarian chain + LangGraph memory


In [ ]:
# Persona gives the bot CHARACTER and SCOPE ("librarian of Alice"); the history
# placeholder is where the graph node will inject this thread's past turns.
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

LIBRARIAN_PERSONA = (
    "You are the helpful librarian of Alice: you know Wonderland deeply, recommend "
    "chapters, keep answers short (2-3 sentences), and never invent plot events."
)
LIBRARIAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", LIBRARIAN_PERSONA),
    MessagesPlaceholder(variable_name="history"),   # the node fills this per thread
    ("human", "{input}"),
])

from langchain_ollama import ChatOllama
base_llm = ChatOllama(model="llama3.2", temperature=0)   # a little warmth is fine here

LLM_LIVE = False
try:
    base_llm.invoke("Reply with the single word: pong")   # one cheap probe up front
    LLM_LIVE = True
except Exception as exc:
    print("[demo skipped] install/start Ollama and run: ollama pull llama3.2")
    print("   detail: %s: %s" % (type(exc).__name__, exc))

def fake_librarian(prompt_value):
    """OFFLINE STUB so the PIPELINE mechanics run anywhere; swap llm back for real answers.

    Mimics the persona deterministically: greets by remembered name when asked,
    recommends chapter one for reading advice, otherwise echoes politely.
    """
    try:
        msgs = prompt_value.to_messages()
    except Exception:
        msgs = []
    last_human = ""
    for m in reversed(msgs):
        if getattr(m, "type", "") == "human":
            last_human = m.content
            break
    low = last_human.lower()
    if "name" in low:
        found = re.findall(r"name is ([A-Za-z]+)",
                           " ".join(getattr(m, "content", "") for m in msgs),
                           flags=re.IGNORECASE)
        if found:
            return "Welcome back to the Alice collection, %s." % found[-1]
        return "I am afraid you have not told me your name yet."
    if "recommend" in low or "read" in low or "start" in low:
        return "Start at Chapter 1, 'Down the Rabbit-Hole' - everything spirals from there."
    return "[offline stub] Noted: \"%s\" (swap llm back for real answers)" % last_human[:80]

if LLM_LIVE:
    llm = base_llm
else:
    from langchain_core.runnables import RunnableLambda
    llm = RunnableLambda(fake_librarian)
    print(">> running with the OFFLINE STUB model for this session")

chatbot_core = LIBRARIAN_PROMPT | llm | StrOutputParser()

# --- the memory layer: a one-node graph plus a checkpointer --------------------
# `MessagesState` carries a `messages` list with an APPEND reducer, so the node
# returns only the new AI message and LangGraph grows the transcript. The
# checkpointer loads/saves that transcript per `thread_id` on every invoke.
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

def librarian_node(state: MessagesState) -> dict:
    """Everything before the newest message is history; the newest is the input."""
    history = state["messages"][:-1]
    latest = state["messages"][-1].content
    return {"messages": [AIMessage(content=chatbot_core.invoke(
        {"history": history, "input": latest}))]}

_builder = StateGraph(MessagesState)
_builder.add_node("librarian", librarian_node)
_builder.add_edge(START, "librarian")
chatbot = _builder.compile(checkpointer=InMemorySaver())

THREAD_IDS = []                           # remembered so we can export transcripts

def guarded_invoke(app, text, config=None):
    """Send one human turn through the graph; returns the reply text or None."""
    try:
        result = app.invoke({"messages": [HumanMessage(content=text)]}, config=config)
        return result["messages"][-1].content
    except Exception as exc:
        print("[demo skipped] install/start Ollama and run: ollama pull llama3.2")
        print("   detail: %s: %s" % (type(exc).__name__, exc))
        return None

print("librarian chatbot assembled:",
      "persona | history | human -> model -> parser, checkpointed per thread")


### Section 3: the capped interactive loop


In [ ]:
# WHY the guards: input() blocks forever under automation, which would hang CI/QA
# runs of this notebook. Three safeties make execution deterministic:
#   MAX_TURNS hard cap, explicit exit commands, EOFError when stdin is closed.
INTERACTIVE = False                       # set True for a real terminal chat
SCRIPTED_INPUTS = ['What made Alice grow so tall?',
                   'And how did she shrink again?',
                   'quit']                # deterministic QA turns
MAX_TURNS = 3                             # learners: raise me for a real chat
EXIT_COMMANDS = {"quit", "exit", "q"}

class _StdinClosed(Exception):
    """Raised by our input shim when no interactive terminal exists."""

def _input_or_script(prompt):
    # Under automated execution there is no stdin: StdinNotImplementedError fires.
    # We translate that into our own sentinel so the loop stays testable.
    try:
        return input(prompt).strip()
    except Exception as exc:              # StdinNotImplementedError lives here
        if type(exc).__name__ == "StdinNotImplementedError":
            raise _StdinClosed
        raise

def chat_loop(thread_id: str, max_turns=MAX_TURNS):
    """Chat loop; returns THIS thread's user/bot transcript list."""
    cfg = {"configurable": {"thread_id": thread_id}}
    THREAD_IDS.append(thread_id)
    transcript = []
    print("=" * 64)
    print("chat thread '%s' - cap %d turns" % (thread_id, max_turns))
    for turn_no in range(1, max_turns + 1):
        try:
            if INTERACTIVE:
                user_text = _input_or_script("you > ")
            else:
                user_text = (SCRIPTED_INPUTS.pop(0)
                             if SCRIPTED_INPUTS else "quit")
                print("you >", user_text)
        except (_StdinClosed, EOFError, KeyboardInterrupt):
            print("[input stream ended] ending session early")
            break
        if not user_text:
            continue
        if user_text.lower() in EXIT_COMMANDS:
            print("librarian > goodbye!")
            break
        reply = guarded_invoke(chatbot, user_text, config=cfg)
        if reply is None:                 # provider failed mid-loop
            break
        print("librarian >", str(reply)[:400])
        transcript.append({"role": "user", "text": user_text})
        transcript.append({"role": "bot", "text": str(reply)})
    else:
        print("(turn cap %d reached)" % max_turns)
    print("=" * 64)
    return transcript

live_transcript = chat_loop("reader-live")
if live_transcript:
    print("captured %d turns from the live thread" % len(live_transcript))
else:
    print("no live turns captured under automation - scripted threads follow")


### Section 4: two scripted threads - separation proof


In [ ]:
# No input() here: we drive the SAME graph directly with different thread_ids, so
# runs are deterministic while still exercising the checkpointer end to end.
def scripted_chat(thread_id, lines):
    """Feed fixed turns through the real graph; returns the reply strings."""
    cfg = {"configurable": {"thread_id": thread_id}}
    THREAD_IDS.append(thread_id)
    replies = []
    print("-" * 64)
    for line in lines:
        out = guarded_invoke(chatbot, line, config=cfg)
        if out is None:
            break
        replies.append(str(out))
        print("[%s] you > %s" % (thread_id, line))
        print("[%s] lib > %s" % (thread_id, str(out)[:200]))
    return replies

scripted_chat("reader-milo", ["Hello! My name is Milo.",
                              "What is my name?",
                              "What should I read first?"])
scripted_chat("reader-june", ["Hi! My name is June.",
                              "What is my name?"])          # must say JUNE, not Milo

def thread_messages(thread_id):
    """Read one thread's stored transcript straight out of the checkpointer."""
    snapshot = chatbot.get_state({"configurable": {"thread_id": thread_id}})
    return snapshot.values.get("messages", []) if snapshot.values else []

print("-" * 64)
print("stored threads:")
for tid in sorted(set(THREAD_IDS)):
    print("   %-13s %d message(s)" % (tid, len(thread_messages(tid))))
print("same questions, different thread_ids, different memories - separation proven.")


### Section 5: transcript export


In [ ]:
# Serialize every thread's checkpointed messages to JSON so learners can diff what
# the model actually SAW versus what it replied.
ROLE_MAP = {"human": "user", "ai": "bot"}          # message.type -> readable role
TRANSCRIPTS = {}
for tid in sorted(set(THREAD_IDS)):
    TRANSCRIPTS[tid] = [{"role": ROLE_MAP.get(m.type, m.type), "text": m.content}
                        for m in thread_messages(tid)]

EXPORT_PATH = DATA / "transcripts.json"
with open(EXPORT_PATH, "w", encoding="utf-8") as fh:
    json.dump(TRANSCRIPTS, fh, indent=2, ensure_ascii=True)   # ASCII-safe file too
print("exported transcripts:")
for tid, entries in TRANSCRIPTS.items():
    print("   %-13s %d entries" % (tid, len(entries)))
print("file:", EXPORT_PATH)


### Summary

- An app is four pieces: persona prompt, guarded model, parser, and a
  LangGraph app with a checkpointer - composed once, reused for every
  thread id forever after.
- `thread_id` is the whole isolation story. It replaces the `session_id` that
  the deprecated `RunnableWithMessageHistory` used, and it is the key the
  checkpointer reads and writes state under.
- Interactive does not mean fragile: MAX_TURNS + exit commands + EOFError
  handling make the same code safe for humans AND automation.
- Scripted double-thread runs are your regression tests for isolation; the
  JSON export - read back out of the checkpointer with `get_state` - is your
  audit trail of every turn stored.
- The capstone keeps exactly this skeleton and upgrades the core from
  "librarian persona" to "RAG over alice.txt with citations".